# CodeX Project - Predictive Modeling

This notebook follows `predictive_modeling_guide.pdf` and builds classification models to predict `price_range`.


In [12]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


In [13]:
# Load engineered dataset
input_path = 'dataset/survey_results_feature_engineered.csv'
df = pd.read_csv(input_path)
print('Dataset shape:', df.shape)
print('Columns:', df.columns.tolist())
print()
print('Target distribution:')
print(df['price_range'].value_counts())


Dataset shape: (29956, 20)
Columns: ['respondent_id', 'gender', 'zone', 'occupation', 'income_levels', 'consume_frequency(weekly)', 'current_brand', 'preferable_consumption_size', 'awareness_of_other_brands', 'reasons_for_choosing_brands', 'flavor_preference', 'purchase_channel', 'packaging_preference', 'health_concerns', 'typical_consumption_situations', 'price_range', 'age_group', 'cf_ab_score', 'zas_score', 'bsi']

Target distribution:
price_range
200-250    9711
150-200    8797
100-150    7793
50-100     3655
Name: count, dtype: int64


## 1) Prepare Features (`X`) and Target (`y`)
- Exclude `respondent_id` and target `price_range` from `X`.


In [14]:
X = df.drop(columns=['respondent_id', 'price_range']).copy()
y = df['price_range'].copy()

print('X shape:', X.shape)
print('y shape:', y.shape)


X shape: (29956, 18)
y shape: (29956,)


## 2) Feature Encoding
- Convert ordered categorical features to ordinal numeric values:
  - `age_group`, `income_levels`, `health_concerns`, `consume_frequency(weekly)`, `awareness_of_other_brands`, `preferable_consumption_size`, `zone`
- One-Hot Encode all remaining categorical feature columns in `X`
- Label Encode target `price_range`


In [15]:
# Ordered categorical feature mappings (feature improvements)
age_group_map = {'18-25': 1, '26-35': 2, '36-45': 3, '46-55': 4, '56-70': 5, '70+': 6}
income_levels_map = {'Not Reported': 0, '<10L': 1, '10L - 15L': 2, '16L - 25L': 3, '26L - 35L': 4, '> 35L': 5}
health_concerns_map = {
    'Low (Not very concerned)': 1,
    'Medium (Moderately health-conscious)': 2,
    'High (Very health-conscious)': 3
}
consume_freq_map = {'0-2 times': 1, '3-4 times': 2, '5-7 times': 3}
awareness_map = {'0 to 1': 1, '2 to 4': 2, 'above 4': 3}
size_map = {'Small (250 ml)': 1, 'Medium (500 ml)': 2, 'Large (1 L)': 3}
zone_map = {'Rural': 1, 'Semi-Urban': 2, 'Urban': 3, 'Metro': 4}

X['age_group'] = X['age_group'].map(age_group_map)
X['income_levels'] = X['income_levels'].map(income_levels_map)
X['health_concerns'] = X['health_concerns'].map(health_concerns_map)
X['consume_frequency(weekly)'] = X['consume_frequency(weekly)'].map(consume_freq_map)
X['awareness_of_other_brands'] = X['awareness_of_other_brands'].map(awareness_map)
X['preferable_consumption_size'] = X['preferable_consumption_size'].map(size_map)
X['zone'] = X['zone'].map(zone_map)

# Target label encoding
y_encoder = LabelEncoder()
y_encoded = y_encoder.fit_transform(y.astype(str))

print('Target classes:', list(y_encoder.classes_))
print('Numeric columns after ordinal mapping:', X.select_dtypes(exclude=['object']).shape[1])


Target classes: ['100-150', '150-200', '200-250', '50-100']
Numeric columns after ordinal mapping: 10


In [16]:
# One-hot encode remaining categorical feature columns
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=False)

# LightGBM does not allow special JSON characters in feature names.
# Normalize all feature names once so every model uses the same matrix.
X_encoded.columns = (
    X_encoded.columns
    .str.replace(r'[^0-9a-zA-Z_]+', '_', regex=True)
    .str.strip('_')
)

print('Encoded feature shape:', X_encoded.shape)
print('Remaining object columns after encoding:', X_encoded.select_dtypes(include=['object']).shape[1])


Encoded feature shape: (29956, 32)
Remaining object columns after encoding: 0


## 3) Data Splitting
- Train/Test split: 75% / 25%
- `random_state=42`


In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y_encoded,
    test_size=0.25,
    random_state=42,
    stratify=y_encoded
)

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)


Train shape: (22467, 32)
Test shape: (7489, 32)


## 4) Model Training and Evaluation
Models:
- Gaussian Naive Bayes
- Logistic Regression
- Support Vector Machine (SVM)
- Random Forest
- XGBoost
- LightGBM

Metrics:
- Accuracy
- Classification Report


In [18]:
models = {
    'Gaussian Naive Bayes': GaussianNB(),
    'Logistic Regression': LogisticRegression(max_iter=3000, random_state=42),
'Support Vector Machine (SVM)': SVC(kernel='rbf', C=1.0, gamma='scale'),
    'Random Forest': RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=1),
    'XGBoost': XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=42,
        objective='multi:softprob',
        eval_metric='mlogloss',
        n_jobs=1,
        verbosity=0
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        random_state=42,
        objective='multiclass',
        n_jobs=1,
        verbose=-1
    )
}
results = []
reports = {}
trained_models = {}
for model_name, model in models.items():
    print()
    print('='*80)
    print(model_name)
    print('='*80)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    report_text = classification_report(
        y_test,
        y_pred,
        target_names=y_encoder.classes_,
        zero_division=0
    )
    print(f"Accuracy: {accuracy:.4f}")
    print(report_text)
    results.append({'Model': model_name, 'Accuracy': accuracy})
    reports[model_name] = report_text
    trained_models[model_name] = model



Gaussian Naive Bayes
Accuracy: 0.5877
              precision    recall  f1-score   support

     100-150       0.50      0.25      0.33      1948
     150-200       0.60      0.42      0.49      2199
     200-250       0.73      0.89      0.80      2428
      50-100       0.42      0.93      0.57       914

    accuracy                           0.59      7489
   macro avg       0.56      0.62      0.55      7489
weighted avg       0.59      0.59      0.56      7489


Logistic Regression
Accuracy: 0.8348
              precision    recall  f1-score   support

     100-150       0.80      0.80      0.80      1948
     150-200       0.79      0.79      0.79      2199
     200-250       0.90      0.91      0.90      2428
      50-100       0.85      0.82      0.83       914

    accuracy                           0.83      7489
   macro avg       0.83      0.83      0.83      7489
weighted avg       0.83      0.83      0.83      7489


Support Vector Machine (SVM)
Accuracy: 0.8463
      

## 5) Model Comparison and Best Model


In [19]:
results_df = pd.DataFrame(results).sort_values(by='Accuracy', ascending=False).reset_index(drop=True)
results_df


,Model,Accuracy
0,LightGBM,0.927761
1,XGBoost,0.922954
2,Random Forest,0.898918
3,Support Vector Machine (SVM),0.846308
4,Logistic Regression,0.834824
5,Gaussian Naive Bayes,0.587662


In [20]:
best_model_name = results_df.loc[0, 'Model']
best_model_accuracy = results_df.loc[0, 'Accuracy']

print('Best model:', best_model_name)
print('Best accuracy:', round(float(best_model_accuracy), 4))


Best model: LightGBM
Best accuracy: 0.9278


In [21]:
# Save model comparison table
results_df.to_csv('dataset/model_comparison_results.csv', index=False)
print('Saved: dataset/model_comparison_results.csv')


Saved: dataset/model_comparison_results.csv


In [22]:
final_summary = {
    'input_rows': int(df.shape[0]),
    'input_columns': int(df.shape[1]),
    'encoded_feature_columns': int(X_encoded.shape[1]),
    'train_rows': int(X_train.shape[0]),
    'test_rows': int(X_test.shape[0]),
    'best_model': best_model_name,
    'best_accuracy': round(float(best_model_accuracy), 4)
}

final_summary


{'input_rows': 29956,
 'input_columns': 20,
 'encoded_feature_columns': 32,
 'train_rows': 22467,
 'test_rows': 7489,
 'best_model': 'LightGBM',
 'best_accuracy': 0.9278}